# OpenCaseLaw API: a practical tour

This notebook demonstrates the independent `opencaselaw` Python client for OpenCaseLaw's public research API. It is not an official OpenCaseLaw client.

Run the setup first, then work through the examples in order: later cells reuse search results, a selected decision, and the tool catalog. Most code cells make live requests. Results and coverage can change between runs; saved outputs show an earlier response, not necessarily the current corpus.

See the [compatibility reference](../docs/api-compatibility.md) for the wrapper inventory, saved contract dated 2026-10-13, and verification limits. This notebook's saved outputs are historical examples, not evidence of current availability or a complete successful run.

Keep request pacing enabled and use non-confidential example queries. The draft, claim, and mock-decision examples send their supplied text to the public service.


## 1. Setup and installation

Run `uv sync` in the repository root, then select that repository's `.venv` as the notebook kernel. After updating the client code, restart the kernel and rerun setup so Python loads the new version.

The setup creates a shared client used throughout this notebook. `preview()` shows model payloads through `.raw`; its `limit` truncates top-level lists, not nested dictionaries or the API response. The final cell closes the shared client.


In [ ]:
# Run `uv sync` from the repo root and select the repo `.venv` as this notebook's kernel.
import opencaselaw

print(f"opencaselaw {opencaselaw.__version__} ready")

In [ ]:
from pprint import pprint
from typing import Any

from opencaselaw import OpenCaseLawAPIError, OpenCaseLawClient

client = OpenCaseLawClient()


def preview(value: Any, keys: list[str] | None = None, limit: int = 3) -> None:
    """Print compact previews of dictionaries, lists, and model objects."""
    if hasattr(value, "raw"):
        value = value.raw
    if isinstance(value, dict):
        selected = value if keys is None else {key: value.get(key) for key in keys}
        pprint(selected)
    elif isinstance(value, list):
        print(f"{len(value)} items")
        for item in value[:limit]:
            preview(item, keys=keys, limit=limit)
    else:
        print(value)


DEFAULT_DECISION_ID = "bger_4A_747_2012"
DEFAULT_BGE_REF = "BGE 140 III 86"

## 2. Configuration and API discovery

Inspect the effective base URL, timeout, and request delay before making calls. Defaults can come from `config.yaml` in the kernel's working directory, or from an explicit `config_path`; a missing file uses package defaults.

A timeout or HTTP error means a request failed, not that the search found nothing. Some examples allow 60 or 120 seconds for a particular request; that gives the service more time but does not guarantee completion.


In [ ]:
print(f"Base URL: {client.base_url}")
print(f"Timeout: {client.timeout}s")
print(f"Rate limit delay: {client.rate_limit_delay}s")

### 2.1 Schemas and research tools

The typed research schema describes selected response shapes; the application schema covers the broader set of HTTP routes. The tool catalog lists additional research capabilities and their current `inputSchema`.

The next cell retrieves the typed schema and catalog without invoking a research tool. Keep `tools_by_name` available for the optional administrative-practice example in section 8.


In [ ]:
research_schema = client.get_openapi(research=True)
print("Typed routes:", list(research_schema["paths"]))
tool_catalog = client.list_tools()
tools_by_name = {tool["name"]: tool for tool in tool_catalog["tools"]}
print("Research tools:", sorted(tools_by_name))

### 2.2 Courts

Use this listing to discover court identifiers for search filters. `list_courts()` preserves the server's response, which can be an array or an object. For parsed court records, the client also offers `list_court_models()`.


In [ ]:
courts = client.list_courts()
preview(courts)

### 2.3 Corpus statistics and scraper health

The statistics example requests the corpus figures for `bger`. Scraper health reports collection status by source; it helps explain coverage gaps but does not establish that a particular search is complete.


In [ ]:
statistics = client.get_statistics(court="bger")
preview(statistics)

In [ ]:
health = client.get_scraper_health()
preview(health)

## 3. Finding and reading decisions

Use docket lookup when you already have a case reference, and topic search when you are exploring a legal question. Keep the returned decision IDs and source links for subsequent retrieval.


### 3.1 Docket lookup and topic search

The first cell looks up a known BGE reference with `exact=True`. Inspect all returned candidates: lookup `total` counts returned hits, and a full page cannot establish uniqueness.

The second cell searches a topic. `q` and `query` are both forwarded when supplied; precedence is server-side. `include_pinpoint=False` skips extra passage lookups. Read `total_is_lower_bound`, `degraded`, and `note` before interpreting the count or ranking; `None` means that metadata was not supplied.


In [ ]:
# Fast lookup for a known citation; inspect all hits for ambiguity.
lookup = client.lookup(DEFAULT_BGE_REF, exact=True, limit=5)
preview(lookup)
print("Lookup total counts returned hits only; a full page does not prove uniqueness.")

In [ ]:
results = client.search_decisions(
    q="Mietrecht Kündigung", limit=5, include_pinpoint=False, request_timeout=60.0
)

count_label = "At least" if results.total_is_lower_bound else "Reported"
print(f"{count_label} {results.total} matches; showing {len(results.results)}")
print("Degraded ranking:", results.degraded, "Note:", results.note)
for decision in results.results:
    print(decision.decision_id, decision.citation_string_de, decision.decision_date)
    print(" ", decision.title)
    print(" ", decision.canonical_url)

### 3.2 Filters, sorting, and publication flags

The first example narrows decisions by court, canton, language, and date, then requests compact results in descending date order. Compact records can omit fields such as titles, so some printed values may be `None`.

The second example selects BGer rulings marked for publication in the official BGE collection. The flag does not mean a BGE number has already been assigned. The old `decision_type` filter is unsupported; express that concept in the query or use a supported court/chamber filter.


In [ ]:
filtered = client.search_decisions(
    query="Arbeitsvertrag Kündigung",
    court="bger",
    canton="CH",
    language="de",
    date_from="2020-01-01",
    date_to="2024-12-31",
    limit=5,
    sort="date_desc",
    fields="compact",
    include_pinpoint=False,
    request_timeout=60.0,
)

for decision in filtered.results:
    print(decision.decision_date, decision.citation_string_de, decision.title)

In [ ]:
# BGer rulings marked for future publication in the official BGE collection.
publication_candidates = client.search_decisions(
    court="bger",
    marked_for_publication=True,
    sort="date_desc",
    limit=3,
    fields="compact",
    include_pinpoint=False,
    request_timeout=60.0,
)
preview(publication_candidates)

### 3.3 Follow the server's continuation

This example retrieves at most two pages. Request the next page only when `has_more` is true and `next_offset` is present, keeping the same search options. Do not calculate the offset from the requested page size: the service can return fewer records.

Reaching `has_more=False` ends the advertised pages; the client cannot establish that these include all relevant decisions in the corpus.


In [ ]:
page_size = 3
page_1 = client.search_decisions(
    query="Datenschutz", limit=page_size, include_pinpoint=False, request_timeout=60.0
)
print("Page 1:", [item.decision_id for item in page_1.results])

# The service can return fewer items than requested; use its continuation offset.
if page_1.has_more is True and page_1.next_offset is not None:
    page_2 = client.search_decisions(
        query="Datenschutz",
        limit=page_size,
        offset=page_1.next_offset,
        include_pinpoint=False,
        request_timeout=60.0,
    )
    print("Page 2:", [item.decision_id for item in page_2.results])
else:
    print("No continuation advertised by the server.")
print("A relevance search with has_more=False is not proof of exhaustive research.")

### 3.4 Fetch a decision for the remaining examples

The next cell selects the first result from the initial topic search, falling back to a fixed example ID when that search is empty. It requests full text but prints only the first 500 characters.

Later structure, citation-graph, and export examples reuse `decision_id`. Examples using `DEFAULT_BGE_REF` refer to a separate fixed citation. The first topic-search hit is illustrative; it is not automatically authority for the sample claims below.


In [ ]:
decision_id = results.results[0].decision_id if results.results else DEFAULT_DECISION_ID
decision = client.get_decision(decision_id, full_text=True)

print(decision.decision_id)
print(decision.court, decision.decision_date, decision.language)
print(decision.title)
print((decision.full_text or "")[:500])

## 4. Structure, headnotes, and passages

These cells retrieve the selected decision's structure, its Regeste (headnote), and a specific Erwägung (reasoning passage). Structure excerpts are shortened; use `get_erwaegung()` for the requested passage text. Not every decision has an indexed headnote or an Erwägung numbered `2.3`; the passage example handles HTTP 404 explicitly.

The final cell asks for up to three passages matching a sample claim using `top_k=3`. Review the returned matches in context. A retrieved passage or a relevance score alone does not establish that the decision supports the claim.


In [ ]:
structure = client.get_structure(decision_id, paragraph_excerpt_chars=300)
preview(structure)

In [ ]:
regeste = client.get_regeste(decision_id)
preview(regeste)

In [ ]:
import httpx

try:
    erwaegung = client.get_erwaegung(decision_id, "2.3")
    preview(erwaegung)
except httpx.HTTPStatusError as error:
    if error.response.status_code != 404:
        raise
    print(f"This decision may not have Erwägung 2.3: {error.response.status_code}")

In [ ]:
relevant = client.find_relevant_erwaegung(
    decision_id,
    claim="Die Kündigung eines Mietvertrags muss nach Treu und Glauben erfolgen.",
    top_k=3,
    request_timeout=120.0,  # This endpoint can exceed the default 30s timeout.
)
preview(relevant)

## 5. Citations, citation graphs, and claim checks

`cite()` asks the server to resolve and format the fixed reference. Its `exists` field concerns reference resolution, not whether the sample legal proposition is correct.

The graph examples use the selected `decision_id`: outgoing citations are references it makes, while incoming citations are references to it. Inspect the separate continuation flags for each direction. Appeal-chain, leading-case, and trend results provide further research context; citation frequency alone does not determine legal authority.


In [ ]:
citation = client.cite(DEFAULT_BGE_REF, pinpoint="2.3", language="de")
print(citation.exists)
print(citation.citation_string_de)
print(citation.canonical_url)
print(citation.rule_statement)
print(citation.pinpoint_valid, citation.pinpoint_status)
if citation.pinpoint_valid is not True:
    print(citation.pinpoint_note, citation.valid_pinpoints)

In [ ]:
citations = client.get_citations(
    decision_id, direction="both", min_confidence=0.3, limit=5, offset=0
)
preview(citations)
print("Continuation:", citations.get("next_offset"))
print("More incoming:", citations.get("incoming_has_more"))
print("More outgoing:", citations.get("outgoing_has_more"))

In [ ]:
appeal_chain = client.get_appeal_chain(decision_id, min_confidence=0.3)
preview(appeal_chain)

In [ ]:
leading_cases = client.find_leading_cases(
    query="Mietrecht Kündigung", court="bger", limit=5, include_pinpoint=False
)
preview(leading_cases)

In [ ]:
trends = client.analyze_trends(query="Datenschutz", court="bger")
preview(trends)

### 5.1 Draft attestation and claim support

The following cells submit short, synthetic examples: first a draft containing a citation, then a claim paired with the selected decision. Inspect the detailed response rather than treating a successful request as a legal verdict. Attestation concerns the supplied draft; it does not establish that no relevant authority is missing.


In [ ]:
attestation = client.attest(draft_text="Gemäss BGE 140 III 86 E. 2.3 ist die Kündigung zu prüfen.")
preview(attestation)

In [ ]:
claim_check = client.verify_claim(
    claim="Die Kündigung eines Mietvertrags darf nicht rechtsmissbräuchlich sein.",
    decision_id=decision_id,
)
preview(claim_check)

## 6. Statutes and legislation

The first examples search federal statute articles and retrieve Art. 41 OR in German. Searching article text and browsing the LexFind legislation catalog answer different questions; both are demonstrated below.

Language filters are explicit choices. Omitting them leaves language selection to the server; inspect the returned edition and source information before relying on an article's wording.


In [ ]:
law_hits = client.search_laws("Schadenersatz", canton="CH", jurisdiction="federal", limit=5)
preview(law_hits)

In [ ]:
law = client.get_law("OR", article="41", language="de")
print(law.abbreviation, law.sr_number, law.title)
for article in law.articles or []:
    print(article.article_num, article.heading)
    print((article.text or "")[:800])
    print("Text status:", article.text_status)

### 6.1 Cantonal laws and federal XML

The next cell attempts a Zürich statute lookup through the cantonal route, then requests Akoma Ntoso XML for a federal article. An unresolved `StG`/`ZH` lookup raises `OpenCaseLawAPIError`, including for an HTTP 200 error response. Inspect its `.payload` for candidates and source identifiers to resolve ambiguity. The XML example handles HTTP 404 when no fragment is available.

The model permits absent article text and a null article collection. Inspect `text_status`: a heading or an empty string is not the article's operative wording. XML availability also varies by edition.


In [ ]:
try:
    cantonal_law = client.get_cantonal_law("StG", "ZH", article="12")
    preview(cantonal_law)
except OpenCaseLawAPIError as error:
    preview(error.payload)
try:
    article_xml = client.get_law_xml("OR", article="41", language="de")
    print(article_xml[:500])
except httpx.HTTPStatusError as error:
    if error.response.status_code != 404:
        raise
    print("No XML fragment is available for this article/edition.")

### 6.2 Amendment references and the legislation catalog

The amendment example asks the service to resolve a sample BBl reference; check whether the response actually resolves it. The next examples search the LexFind catalog, retrieve a record only if a result supplies an ID, and browse reported federal-law changes.

Catalog results and changes are research aids. Check the identified act, edition, and effective dates rather than assuming a search hit supplies the applicable wording.


In [ ]:
amendment = client.resolve_amendment_ref(ref_type="BBl", year=2020, page=1)
preview(amendment)

In [ ]:
legislation_hits = client.search_legislation(
    query="Datenschutz",
    canton="CH",
    limit=5,
    active_only=True,
    search_in_content=False,
)
preview(legislation_hits)

In [ ]:
# Use a LexFind ID from the previous result if the response exposes one.
lexfind_id = None
items = legislation_hits.get("results") or legislation_hits.get("items") or []
if items and isinstance(items[0], dict):
    lexfind_id = items[0].get("lexfind_id") or items[0].get("id")

if lexfind_id is not None:
    legislation = client.get_legislation(int(lexfind_id), include_versions=False)
    preview(legislation)
else:
    print("No LexFind ID found in the search response preview.")

In [ ]:
changes = client.get_legislation_changes(canton="CH")
preview(changes)

## 7. Commentaries, legislative materials, and scholarship

These examples distinguish several sources of research context: commentary on an article, a combined doctrine response, legislative materials, and scholarly publications. Read source references and dates alongside the returned text.

The first cells search commentaries, retrieve commentary on Art. 41 OR, and request a doctrine response for that topic. The German language filter in the commentary examples is explicit; omitting it leaves language selection to the server.


In [ ]:
commentary_hits = client.search_commentaries(
    "Schadenersatz", abbreviation="OR", language="de", limit=5
)
preview(commentary_hits)

In [ ]:
commentary = client.get_commentary("OR", article="41", language="de")
preview(commentary)

In [ ]:
doctrine = client.get_doctrine("Art. 41 OR Schadenersatz")
preview(doctrine)

### 7.1 Legislative materials and article history

Search materials by topic or law, retrieve material linked to an article, and explore article purpose and history. Botschaft search uses publication-year bounds; they are not bounds on when the resulting law was in force.

The Botschaft example omits a language filter, leaving language selection to the server. Inspect each source reference and passage before quoting; coverage varies by law and period.


In [ ]:
materials = client.search_materialien("Haftung", law_code="OR", limit=5)
preview(materials)

In [ ]:
article_materials = client.get_materialien("OR", article="41")
preview(article_materials)

In [ ]:
purpose = client.get_article_purpose("220", "41", language="de", max_paragraphs=5)
preview(purpose)

In [ ]:
# Leave language selection to the server; year bounds refer to publication.
botschaft_hits = client.search_botschaft("Schadenersatz", year_min=2000, year_max=2026, limit=5)
preview(botschaft_hits)

In [ ]:
history = client.get_article_history("220", "41", language="de", leading_cases_limit=5)
preview(history)

### 7.2 Scholarship

Inspect the source and license catalogs, then search publications by topic and publication year. Author, source, type, and language filters can also support a bibliographic browse. The example retrieves metadata for the first returned `pub_id`; the extra full-text request remains commented out.

Preserve the returned `attributions` and `license_usage` when reusing results. The saved catalog includes closed-access and metadata-only records; publication metadata does not establish full-text availability or permission to reuse it. The following cell explores scholarship citing a statute or decision and reports citation-bridge statistics; absent links can reflect coverage gaps.


In [ ]:
scholarship_sources = client.list_scholarship_sources()
preview(scholarship_sources)
license_catalog = client.get_scholarship_licenses()
preview(license_catalog)
scholarship = client.search_scholarship("Mietrecht", year_min=2020, sort="year", limit=3)
preview(scholarship)
publications = scholarship.get("results", [])
if publications:
    publication_id = publications[0]["pub_id"]
    publication = client.get_scholarship(publication_id)
    preview(publication)
    # Optional: retrieve text only after reviewing the source's reuse terms.
    # full_text = client.get_scholarship_full_text(publication_id, request_timeout=120.0)
    # preview(full_text)

In [ ]:
statute_scholarship = client.find_scholarship_citing_statute("220", article="41", limit=3)
preview(statute_scholarship)
decision_scholarship = client.find_scholarship_citing_decision(decision_id, limit=3)
preview(decision_scholarship)
preview(client.get_scholarship_citation_stats())

## 8. Additional research helpers

This section first inspects an optional tool, then demonstrates case briefs, teaching questions, and a mock-decision outline. These calls serve different purposes; inspect the source material behind any generated analysis.


### 8.1 Administrative practice through the tool bridge

The next cell displays the current `search_practice` input schema if the deployment advertises that tool. Its execution is commented out. To try it, review the schema and uncomment the call with the sample query or your own non-confidential terms.

`call_tool()` raises `OpenCaseLawToolError` when a tool returns `_is_error: true` inside an HTTP 200 response. Its `.payload` retains the error details. Ordinary HTTP failures still raise `httpx.HTTPStatusError`.


In [ ]:
if "search_practice" in tools_by_name:
    preview(tools_by_name["search_practice"]["inputSchema"])
    # practice = client.call_tool(
    #     "search_practice", {"query": "Mehrwertsteuer", "limit": 3}, request_timeout=120.0
    # )
    # preview(practice)
else:
    print("This deployment does not advertise search_practice.")

### 8.2 Case briefs, teaching questions, and mock decisions

These cells request a brief for the fixed BGE reference, a teaching question about contract law, and an outline based on fictional facts. They make live requests when run. The saved mock-decision response asks for clarification; an outline is not guaranteed. The mock-decision call allows 120 seconds because it can be slow.

Treat the outputs as starting points for reading and verification. The sample facts do not describe the selected search result, and a mock outline is not an actual court decision.


In [ ]:
case_brief = client.get_case_brief(DEFAULT_BGE_REF)
preview(case_brief)

In [ ]:
exam_question = client.generate_exam_question("Vertragsrecht")
preview(exam_question)

In [ ]:
# Research-only: sends facts/question to the public API; do not use confidential facts or treat output as legal advice.
mock = client.mock_decision(
    facts="Eine Mieterin kündigt nach einer strittigen Nebenkostenabrechnung fristlos.",
    question="Welche zivilrechtlichen Fragen stellen sich?",
    preferred_language="de",
    limit=3,
    request_timeout=120.0,
)
preview(mock)

## 9. Exports and feeds

The export cell retrieves Word and PDF versions of the selected decision plus BibTeX and RIS citation records. It holds them in memory and prints sizes or short previews. The file-writing example is commented out; uncomment it only when you want local copies.

The final cell retrieves the BGer Atom feed and prints a prefix of the XML. A feed is a view of recent entries, not a complete corpus export.


In [ ]:
docx_bytes = client.export_docx(decision_id)
pdf_bytes = client.export_pdf(decision_id)
bib_text = client.export_bib(decision_id)
ris_text = client.export_ris(decision_id)

print(f"DOCX bytes: {len(docx_bytes):,}")
print(f"PDF bytes: {len(pdf_bytes):,}")
print(bib_text[:500])
print(ris_text[:500])

In [ ]:
# Uncomment to save exports locally. Generated files are intentionally not tracked.
# from pathlib import Path

# output_dir = Path("downloads")
# output_dir.mkdir(exist_ok=True)
# (output_dir / f"{decision_id}.docx").write_bytes(docx_bytes)
# (output_dir / f"{decision_id}.pdf").write_bytes(pdf_bytes)
# (output_dir / f"{decision_id}.bib").write_text(bib_text, encoding="utf-8")
# (output_dir / f"{decision_id}.ris").write_text(ris_text, encoding="utf-8")

In [ ]:
feed = client.atom_feed("bger")
print(feed[:1000])

## 10. Retrieve an integrity proof

This call retrieves the selected decision's inclusion-proof payload. Displaying it does not verify it: cryptographic verification requires checking the proof against the relevant published root. An inclusion proof concerns corpus inclusion, not the correctness of a legal conclusion.


In [ ]:
integrity = client.get_integrity_proof(decision_id)
preview(integrity)

## 11. Client lifetime and request pacing

A `with` block closes its client automatically, including when a request raises an exception. The second example uses `try/finally` to close a separate client after a deliberately slower search.

The request delay is per client instance, not a shared limiter across notebooks or processes. Coordinate concurrent jobs so their combined traffic remains modest. These synchronous examples make requests sequentially; failed requests are not automatically retried.


In [ ]:
with OpenCaseLawClient(timeout=60.0, rate_limit_delay=0.2) as managed_client:
    managed_results = managed_client.search_decisions(
        query="Bundesgericht",
        court="bger",
        limit=3,
        fields="compact",
    )
    print(
        "Reported count:",
        managed_results.total,
        "Lower bound:",
        managed_results.total_is_lower_bound,
    )

print("Client closed after leaving the context manager")

In [ ]:
# Long-running batch jobs should keep the default 0.2s delay or use a larger value.
slow_client = OpenCaseLawClient(timeout=30.0, rate_limit_delay=1.0)
try:
    sample = slow_client.search_decisions(query="Test", limit=1)
    print(f"Sample result count: {sample.total}")
finally:
    slow_client.close()

## 12. Close the shared client

Run the final cell when you finish, including after an earlier example fails. If you stop before reaching it, run `client.close()` yourself. To resume API calls after closing, rerun the setup that creates the client.

For another research session, start with your own query or reference, retain the returned IDs and source links, and inspect pagination and coverage notes. The [README](../README.md) and [API compatibility reference](../docs/api-compatibility.md) describe the wrapper options; this package has no CLI or offline research workflow.


In [ ]:
client.close()